ANX-CBT - rsfMRI


In [ ]:
# Nicholas Bustos
# August 2026
# 
# https://gemini.google.com/app/9944ff77d01210bf

## Hypotheses:

**BASELINE:**
Does the UF-fa mediate vmpfc2amyg connectivity ? 

1. H1: Lower FA → higher anxiety (PARS)  
Moderated by age/sex/pub ?  
2. H2: Lower FA → greater amyg & lesser pfc [functional (EFSAT)]:
All Faces vs Fixation (Negative faces vs Fixation)
whole brain activation 
Use finding as starting point for ROI analyses
ROIs (amygdala & large PFC mask)
3. H3: Lower FA → Lower amyg-vmpfc tb-connectivity:
Regress out task events  
Seed2voxel (with large PFC mask)
ROIs:
Amyg-vmpfc
amyg-PFC(larger mask)


### Notes:

- 
- 



In [ ]:
import os
import re
import glob
import warnings
warnings.filterwarnings('ignore') # Suppresses 'Mean of empty slice' warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multitest import multipletests

In [ ]:
#final_df.to_excel("UF_EFSAT_NB.xlsx")

### Variables


- MAIN:
- tx (CBT/RMT)
- group_dg (Anx HC)
- group (AD / HC)
- scanage (years)
- dev_stage (7 - 11 - 14)
- sex (1 = male / 2 = female)
- pds_score ( Pubertal Development Scale)
- pars0 (Baseline PARS)
- pars12 (Post PARS)
- agexpars (int)


- UF:
- ad_l , ad_r
- rd_l , rd_r
- dti_fa_l , dti_fa_r
- md_l , md_r
- qa_l , qa_r
- GEOMETRY:
- volumemm3_l
- mean_lengthmm_l
- branch_volumemm3_r
- curl_r
- total_area_of_end_regionsmm2_l


- EFSAT:
- totalaccvalid
- congacc
- incongtotal
- efsat_fear_hits (number of times correctly matched fearful faces)
- efsat_happy_com (number of times selected the wrong face during a happy trial)
- efsat_angry_om  (number of missed angry trials)
- efsat_angry_rthits (average speed of correct responses during angry face trials)
- efsat_shapefear_rtcom (average speed of incorrect (commission) responses during the shape control trials paired with the fear blocks)
- efsat_alltrials_acc (overall accuracy percentage across the entire fMRI run (faces and shapes combined)
  

### Start

In [ ]:
# ==========================================
# CELL 1: SETUP & ATLAS AUGMENTATION
# ==========================================
# This cell imports the required libraries, loads the Schaefer 356-node atlas, and 
# applies custom network designations to map the subcortical and cerebellar nodes 
# to the Yeo 7-network and 17-network schemas.

matrices_dir = os.environ.get('RSX_MATRICES_DIR', 'data/Custom_4S356_matrices')
atlas_path = os.environ.get('RSX_ATLAS_TSV', 'data/atlas-4S356Parcels_dseg.tsv')

atlas_df = pd.read_csv(atlas_path, sep='\t')

# 1. Subcortical: Rows 301 to 346
atlas_df.loc[300:345, 'network_label'] = 'Subcort'
atlas_df.loc[300:345, 'network_label_17network'] = 'Subcort'

# 2. Cerebellar: Rows 347 to 356
atlas_df.loc[346:355, 'network_label'] = 'Cerebellum'
atlas_df.loc[346:355, 'network_label_17network'] = 'Cerebellum'

labels_7 = atlas_df['network_label'].values
labels_17 = atlas_df['network_label_17network'].values

print("Atlas augmentation complete.")
print(f"9-Network Unique Labels: {np.unique([l for l in labels_7 if pd.notna(l)])}")
print(f"19-Network Unique Labels: {np.unique([l for l in labels_17 if pd.notna(l)])}")

In [ ]:
# ==========================================
# CELL 2: CORE COMPUTATION FUNCTIONS
# ==========================================
# This cell defines the mathematical engines used to calculate within-network means, 
# between-network means, and seed-to-network integration. It uses Pandas-native 
# boolean masking (.where) to safely drop the diagonal without read-only memory errors.

def compute_network_connectivity(mat, labels, suffix=""):
    unique_labels = sorted([l for l in np.unique(labels) if pd.notna(l)])
    results = {}
    
    mask = ~np.eye(mat.shape[0], dtype=bool)
    mat_nodiag = mat.where(mask, np.nan)
    
    for i, net1 in enumerate(unique_labels):
        idx1 = (labels == net1)
        for j, net2 in enumerate(unique_labels):
            if i <= j: 
                idx2 = (labels == net2)
                if i == j: 
                    sub_mat = mat_nodiag.iloc[idx1, idx2].values
                    results[f'{net1}_within{suffix}'] = np.nanmean(sub_mat)
                else: 
                    sub_mat = mat.iloc[idx1, idx2].values
                    results[f'{net1}_X_{net2}{suffix}'] = np.nanmean(sub_mat)
                    
    return results

def compute_roi_to_network(mat, labels, atlas_df, rois_of_interest, suffix=""):
    unique_labels = sorted([l for l in np.unique(labels) if pd.notna(l)])
    results = {}
    
    for roi in rois_of_interest:
        roi_idx = atlas_df.index[atlas_df['label'] == roi].tolist()[0]
        roi_data = mat.iloc[roi_idx, :]
        
        for net in unique_labels:
            net_idx = (labels == net).copy()
            net_idx[roi_idx] = False 
            results[f'{roi}_to_{net}{suffix}'] = np.nanmean(roi_data.iloc[net_idx])
            
    return results

In [ ]:
# ==========================================
# CELL 3: MAIN INGESTION LOOP (Build df0)
# ==========================================
# This cell iterates through all raw .csv matrices, extracts subject metadata 
# (ID, Visit, Run) from filenames, calculates all macro- and micro-scale features, 
# and compiles them into the foundational dataframe df0.

all_matrices = glob.glob(os.path.join(matrices_dir, '*.csv'))
print(f"Found {len(all_matrices)} matrices to process.")

target_rois = ['LH_Hippocampus', 'LH_Amygdala', 'RH_Hippocampus', 'RH_Amygdala']
data_rows = []

for file_path in all_matrices:
    fname = os.path.basename(file_path)
    
    match = re.search(r'kdf16anx(\d+).*?run-(\d+)', fname, re.IGNORECASE)
    if not match:
        continue
        
    full_digits = match.group(1)
    run = int(match.group(2))
    
    if len(full_digits) >= 5:
        sub_id = full_digits[:4]
        visit = int(full_digits[4])
    else:
        sub_id = full_digits
        visit = np.nan
        
    row_data = {'ID': sub_id, 'Visit': visit, 'Run': run, 'Filename': fname}
    
    mat = pd.read_csv(file_path, index_col=0)
    
    if mat.shape != (356, 356):
        continue
        
    row_data.update(compute_network_connectivity(mat, labels_7, suffix="_7net"))
    row_data.update(compute_network_connectivity(mat, labels_17, suffix="_17net"))
    
    row_data.update(compute_roi_to_network(mat, labels_7, atlas_df, target_rois, suffix="_7net"))
    row_data.update(compute_roi_to_network(mat, labels_17, atlas_df, target_rois, suffix="_17net"))
    
    data_rows.append(row_data)

df0 = pd.DataFrame(data_rows)
print(f"Processing Complete! Extracted {len(df0.columns) - 4} features per matrix.")

In [ ]:
# ==========================================
# CELL 3.5: RUN COUNT AUDIT & FILTERING
# ==========================================
# This cell audits how many scan runs exist per subject at each visit, 
# prints the counts of subjects with and without the full 2-run complement, 
# and filters the dataframe to retain only data from subjects/visits with exactly 2 runs.

# Count unique runs per subject and visit
run_counts = df0.groupby(['ID', 'Visit'])['Run'].nunique().reset_index(name='num_runs')

# Summarize frequencies of run counts
counts_summary = run_counts['num_runs'].value_counts().sort_index()

print("--- Run Count Audit ---")
for num_runs, freq in counts_summary.items():
    print(f"Subjects/Visits with {num_runs} run(s): {freq}")

# Filter df0 to keep only subjects/visits with exactly 2 runs
valid_groups = run_counts[run_counts['num_runs'] == 2][['ID', 'Visit']]
df0 = pd.merge(df0, valid_groups, on=['ID', 'Visit'], how='inner')

print(f"\nFiltered df0 shape (retaining only 2-run entries): {df0.shape}")

In [ ]:
# ==========================================
# CELL X: VISIT OVERLAP AUDIT
# ==========================================
# This cell audits the cohort's longitudinal structure by identifying 
# how many unique subjects have scans for Visit 1 only, Visit 2 only, or both visits.


# Ensure we reference the latest cleaned dataframe (dfm5)
id_col = 'ID' # Adjust if your ID column has a different name

v1_ids = set(df0[df0['Visit'] == 1][id_col].unique())
v2_ids = set(df0[df0['Visit'] == 2][id_col].unique())

only_v1 = v1_ids - v2_ids
only_v2 = v2_ids - v1_ids
both_visits = v1_ids.intersection(v2_ids)

print("--- Longitudinal Visit Audit ---")
print(f"Total Unique Subjects:        {len(v1_ids.union(v2_ids))}")
print(f"Subjects with Visit 1 Only:   {len(only_v1)}")
print(f"Subjects with Visit 2 Only:   {len(only_v2)}")
print(f"Subjects with BOTH Visits:    {len(both_visits)}")

In [ ]:
# ==========================================
# CELL 4: RUN AGGREGATION (Merge Runs)
# ==========================================
# This cell isolates the numeric functional connectivity features within df0 
# and computes the average between multiple scan runs for each subject at 
# each visit, outputting the finalized dataframe df_merged_runs.

# Identify columns that can be averaged from the starting dataframe (df0)[cite: 8]
numeric_cols = df0.select_dtypes(include=[np.number]).columns.tolist()

# Prevent metadata columns from being averaged alongside connectivity values
for col in ['Run', 'Visit', 'ID']:
    if col in numeric_cols:
        numeric_cols.remove(col) 

# Group by ID and Visit and calculate the mean for all connectivity features[cite: 8]
df_merged_runs = df0.groupby(['ID', 'Visit'])[numeric_cols].mean().reset_index()

print(f"Final Aggregated Dataframe Shape: {df_merged_runs.shape}")
print(df_merged_runs.head())

# Save for future use[cite: 8]
# df_merged_runs.to_csv("Master_RSFC_Features.csv", index=False)

In [ ]:
df0_path = os.environ.get('RSX_OUTPUT_PATH', 'outputs/df0.xlsx')
df0.to_excel(df0_path, index=False)

### ROI 2 Network

In [ ]:
# ==========================================
# CELL 5: ROI TO NETWORK MODULE (Dataframe-Based)
# ==========================================
# This cell isolates the pre-calculated Left and Right Amygdala-to-LimbicB 
# features from the existing master dataframe, and computes the Bilateral 
# Amygdala integration by averaging the two hemispheres. It outputs these 
# into a new, streamlined dataframe.

# Define the target columns that already exist in your aggregated dataframe 
lh_col = 'LH_Amygdala_to_LimbicB_17net'
rh_col = 'RH_Amygdala_to_LimbicB_17net'
bilateral_col = 'Bilateral_Amygdala_to_LimbicB_17net'

# Create a new dataframe isolating the identifiers and the specific unilateral features
# (Swap 'df_merged_runs' with 'dfm5' if you are using the motion-scrubbed dataset)
df_roi_network = df_merged_runs[['ID', 'Visit', lh_col, rh_col]].copy()

# Calculate the Bilateral Amygdala to Limbic-B connectivity (mean of LH and RH)
df_roi_network[bilateral_col] = df_roi_network[[lh_col, rh_col]].mean(axis=1)

print(f"Created ROI-to-Network dataframe with {len(df_roi_network)} rows.")
print(df_roi_network.head())

### Developmental Stages

### H1: The structural integrity of the uncinate fasciculus (dti_fa) is not a significant predictor of baseline anxiety severity 